### Preprocessing

In [1]:
import pandas as pd
data = pd.read_csv('~/ML/HCV/NS3_4A_working/NS3_4A_descriptors.csv')
data = data.dropna()

In [2]:
from sklearn.feature_selection import SelectFromModel
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Lasso
import numpy as np

X = data.drop(columns=['Name', 'pIC50'])
y = data['pIC50']
y = y.str.replace(',', '.').astype(float)

In [3]:
X

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,APC2D10_I_I,APC2D10_I_B,APC2D10_I_Si,APC2D10_I_X,APC2D10_B_B,APC2D10_B_Si,APC2D10_B_X,APC2D10_Si_Si,APC2D10_Si_X,APC2D10_X_X
0,0,-0.9857,0.971604,101.9567,66.456169,0,0,62,29,33,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,-5.0610,25.613721,159.5386,103.930064,0,0,97,49,48,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1,0.3347,0.112024,138.1279,76.516204,0,0,67,39,28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1,0.0961,0.009235,133.8775,73.954204,0,0,65,37,28,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1,0.0160,0.000256,101.6093,59.082239,0,0,53,30,23,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,0,-1.9903,3.961294,195.6358,115.133271,0,0,99,52,47,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
491,1,-0.3527,0.124397,96.8931,57.569032,0,0,53,29,24,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
492,3,-4.5390,20.602521,194.3304,122.812822,0,0,112,58,54,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
493,3,-4.6782,21.885555,188.8104,121.672822,0,0,112,58,54,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
categorial_col = [col for col in X.columns if X[col].nunique() == 2 and int(X[col].max())==1]
numeric_col = [col for col in X.columns if not(X[col].nunique() == 2 and int(X[col].max())==1)]
print(len(categorial_col))
print(len(numeric_col))

4773
13194


In [6]:
X[categorial_col[0]].unique()

array([0., 1.])

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_col),
        ('cat', 'passthrough', categorial_col) 
    ])

X_scaled = preprocessor.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=numeric_col + categorial_col)

In [8]:
X_scaled

,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,nHeavyAtom,nH,...,APC2D6_S_S,APC2D6_S_Cl,APC2D7_N_Cl,APC2D7_O_S,APC2D8_N_Cl,APC2D9_N_S,APC2D9_N_Cl,APC2D9_S_Cl,APC2D9_S_X,APC2D10_N_Cl
0,-0.396230,0.559986,-0.764214,-3.082742,-2.326065,0.0,0.0,-2.013051,-2.878635,-1.343991,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2.061909,-1.253057,1.301075,-1.234755,-0.776459,0.0,0.0,-0.555490,-0.704986,-0.421434,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.832840,1.147413,-0.836257,-1.921893,-1.910066,0.0,0.0,-1.804828,-1.791811,-1.651510,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.832840,1.041263,-0.844872,-2.058302,-2.016009,0.0,0.0,-1.888117,-2.009176,-1.651510,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.832840,1.005628,-0.845624,-3.093892,-2.630989,0.0,0.0,-2.387853,-2.769953,-1.959029,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
482,-0.396230,0.113053,-0.513644,-0.076280,-0.313188,0.0,0.0,-0.472200,-0.378938,-0.482938,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
483,0.832840,0.841598,-0.835220,-3.245250,-2.693563,0.0,0.0,-2.387853,-2.878635,-1.897525,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
484,3.290978,-1.020827,0.881079,-0.118174,0.004374,0.0,0.0,0.069180,0.273157,-0.052411,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
485,3.290978,-1.082755,0.988612,-0.295329,-0.042767,0.0,0.0,0.069180,0.273157,-0.052411,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Feature selection

In [10]:
# 1. Decision Tree Regressor
print('DT')
dt = DecisionTreeRegressor(random_state=42)
selector_dt = SelectFromModel(estimator=dt, max_features=50) 
selector_dt.fit(X_scaled, y)
selected_features_dt = selector_dt.get_support()

# 2. Random Forest Regressor
print('RF')
rf = RandomForestRegressor(n_estimators=100, random_state=42)
selector_rf = SelectFromModel(estimator=rf, max_features=50)
selector_rf.fit(X_scaled, y)
selected_features_rf = selector_rf.get_support()

print('GB')
# 3. Gradient Boosting Regressor
gbr = GradientBoostingRegressor(n_estimators=100, random_state=42)
selector_gbr = SelectFromModel(estimator=gbr, max_features=50)
selector_gbr.fit(X_scaled, y)
selected_features_gbr = selector_gbr.get_support()

DT
RF
GB


In [11]:
my_features_NS3_4A = pd.DataFrame()
my_features_NS3_4A['DT'] = X.columns[selected_features_dt]
my_features_NS3_4A['RF'] = X.columns[selected_features_rf]
my_features_NS3_4A['GB'] = X.columns[selected_features_gbr]

print(my_features_NS3_4A)
my_features_NS3_4A.to_csv('my_new_features_NS3_4A.csv')

               DT                RF                GB
0          AATS5m            ATSC5m            AATS8s
1          AATS6v            ATSC5e            ATSC7c
2          AATS6p            ATSC1p            ATSC5e
3          ATSC0c            ATSC0s            ATSC0s
4          ATSC7c            ATSC7s            ATSC7s
5          ATSC3p           AATSC7s           AATSC5e
6         AATSC5m            MATS7c            MATS6p
7         AATSC3v            MATS5e            MATS7s
8         AATSC1e            MATS7s            GATS2v
9         AATSC6e            GATS2v           SM1_Dze
10        AATSC8s            GATS5i           VR2_Dzs
11         MATS2e        SpMax2_Bhm        SpMax1_Bhm
12         GATS2m        SpMax3_Bhm        SpMax2_Bhm
13         GATS7s        SpMin8_Bhm        SpMax2_Bhe
14        VE1_Dzv        SpMax2_Bhv        SpMax3_Bhe
15        SM1_Dzp        SpMax3_Bhv        SpMin8_Bhe
16        VE2_Dzp        SpMax2_Bhe        SpMax8_Bhp
17     SpMin8_Bhm        SpM